In [1]:
# /// script
# require-python = ">=3.11"
# dependencies = ["duckdb", "ipywidgets", "jupysql", "matplotlib", "numpy", "pandas", "tqdm", "zstandard"]
# ///

# Looking up process tree paths

Just take a look at the way process tree paths can be accessed in ACME4.
The good news: you don't have to walk through the `process` table joining between `pid_hash` and `parent_pid_hash` like a 1990's peasant.
The bad news: the Wintap sensor made many mistakes while coalescing these paths,
so sensor noise ahoy.

In [1]:
import duckdb
import logging as lg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm, trange
import warnings

In [2]:
tqdm.pandas()

In [3]:
lg.basicConfig(level=lg.INFO, format="%(asctime)-8s | %(levelname)-8s | %(name)-24s | %(message)s", datefmt="%H:%M:%S")
LOG = lg.getLogger("notebook")

In [32]:
pd.set_option("display.html.use_mathjax", False)
pd.set_option("display.max_colWidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.min_rows", 20)
pd.set_option("styler.html.mathjax", False)

In [5]:
root_acme4 = Path("/data/ACME4/stdview-20240819-20240923")  # Replace with your own data location!
db = duckdb.connect()
for path in root_acme4.glob("*.parquet"):
    name_view = path.with_suffix("").name
    LOG.debug(f"Add view {name_view} over {path}")
    db.sql(f"create or replace view {name_view} as select * from '{path}'")

In [6]:
with warnings.catch_warnings(action="ignore", category=SyntaxWarning):
    %load_ext sql
%sql db --alias duckdb
%config SqlMagic.displaycon=False
%config SqlMagic.autopandas=True

Tip: You may define configurations in /work/home/hamelin/acme4-eda/pyproject.toml or /work/home/hamelin/.jupysql/config.

Did not find user configurations in /work/home/hamelin/acme4-eda/pyproject.toml.

The process tree leaf-to-root paths are stored in the `process_path` table.

In [7]:
%%sql
select *
from process_path
limit 10

,agent_id,hostname,pid_hash,os_pid,process_name,process_path,level,parent_pid_hash,parent_os_pid,seq,ptree,ptree_list,ptree_list_tuples,max_level
0,fe437745-98c3-4d17-8e19-b8ba145caced,ACME-DC1,00076962A860658B5C74574915AAD94C,1932,conhost.exe,c:\windows\system32\conhost.exe,0,AF9348E82E1CDEF72F7E912E0212F56D,2536,1,=conhost.exe,[00076962A860658B5C74574915AAD94C],"[{'pid_hash': '00076962A860658B5C74574915AAD94C', 'process_name': 'conhost.exe'}]",0
1,fe437745-98c3-4d17-8e19-b8ba145caced,ACME-DC1,000C6DD334EAC0C94D4AB0147CB66823,2684,conhost.exe,c:\windows\system32\conhost.exe,0,76BB31D9346CE2F9DE9569E5C1C27EB2,2468,1,=conhost.exe,[000C6DD334EAC0C94D4AB0147CB66823],"[{'pid_hash': '000C6DD334EAC0C94D4AB0147CB66823', 'process_name': 'conhost.exe'}]",0
2,fe437745-98c3-4d17-8e19-b8ba145caced,ACME-DC1,0017CF8579D1B0A203230C4445486C6A,7696,wmic.exe,c:\windows\system32\wbem\wmic.exe,5,200B9DB9449A2F74C1999BD921A616C5,820,6,=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe,"[0017CF8579D1B0A203230C4445486C6A, 509D73E59EA32A053D878E396ADEEEA7, 18325228D5FD390F6FEF7E41190D82F3, 200B9DB9449A2F74C1999BD921A616C5, E348B38A38A42696061DD2CE30F52B62, 916C854F45C9A21E9D80F58E975BAB3A]","[{'pid_hash': '0017CF8579D1B0A203230C4445486C6A', 'process_name': 'wmic.exe'}, {'pid_hash': '509D73E59EA32A053D878E396ADEEEA7', 'process_name': 'ssm-agent-worker.exe'}, {'pid_hash': '18325228D5FD390F6FEF7E41190D82F3', 'process_name': 'amazon-ssm-agent.exe'}, {'pid_hash': '200B9DB9449A2F74C1999BD921A616C5', 'process_name': 'services.exe'}, {'pid_hash': 'E348B38A38A42696061DD2CE30F52B62', 'process_name': 'wininit.exe'}, {'pid_hash': '916C854F45C9A21E9D80F58E975BAB3A', 'process_name': 'svchost.exe'}]",5
3,fe437745-98c3-4d17-8e19-b8ba145caced,ACME-DC1,001999BCC533CC1B0FE072CA9BB692B9,9932,conhost.exe,c:\windows\system32\conhost.exe,0,F1FB902589A7DCF44659B0577841385D,816,1,=conhost.exe,[001999BCC533CC1B0FE072CA9BB692B9],"[{'pid_hash': '001999BCC533CC1B0FE072CA9BB692B9', 'process_name': 'conhost.exe'}]",0
4,fe437745-98c3-4d17-8e19-b8ba145caced,ACME-DC1,001FFD625FDE57A058ABD9EA10E208C5,8920,conhost.exe,c:\windows\system32\conhost.exe,0,D58F28C07DA334C2FDFDA0ED102F7C36,9892,1,=conhost.exe,[001FFD625FDE57A058ABD9EA10E208C5],"[{'pid_hash': '001FFD625FDE57A058ABD9EA10E208C5', 'process_name': 'conhost.exe'}]",0
5,fe437745-98c3-4d17-8e19-b8ba145caced,ACME-DC1,0024EDDD1E7D68B26787944479F7B056,2948,wmic.exe,c:\windows\system32\wbem\wmic.exe,5,200B9DB9449A2F74C1999BD921A616C5,820,6,=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe,"[0024EDDD1E7D68B26787944479F7B056, 509D73E59EA32A053D878E396ADEEEA7, 18325228D5FD390F6FEF7E41190D82F3, 200B9DB9449A2F74C1999BD921A616C5, E348B38A38A42696061DD2CE30F52B62, 916C854F45C9A21E9D80F58E975BAB3A]","[{'pid_hash': '0024EDDD1E7D68B26787944479F7B056', 'process_name': 'wmic.exe'}, {'pid_hash': '509D73E59EA32A053D878E396ADEEEA7', 'process_name': 'ssm-agent-worker.exe'}, {'pid_hash': '18325228D5FD390F6FEF7E41190D82F3', 'process_name': 'amazon-ssm-agent.exe'}, {'pid_hash': '200B9DB9449A2F74C1999BD921A616C5', 'process_name': 'services.exe'}, {'pid_hash': 'E348B38A38A42696061DD2CE30F52B62', 'process_name': 'wininit.exe'}, {'pid_hash': '916C854F45C9A21E9D80F58E975BAB3A', 'process_name': 'svchost.exe'}]",5
6,fe437745-98c3-4d17-8e19-b8ba145caced,ACME-DC1,00298EB1C6BC14848306E4670ED2B9E4,5596,wmic.exe,c:\windows\system32\wbem\wmic.exe,5,200B9DB9449A2F74C1999BD921A616C5,820,6,=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe,"[00298EB1C6BC14848306E4670ED2B9E4, 509D73E59EA32A053D878E396ADEEEA7, 18325228D5FD390F6FEF7E41190D82F3, 200B9DB9449A2F74C1999BD921A616C5, E348B38A38A42696061DD2CE30F52B62, 916C854F45C9A21E9D80F58E975BAB3A]","[{'pid_hash': '00298EB1C6BC14848306E4670ED2B9E4', 'process_name': 'wmic.exe'}, {'pid_hash': '509D73E59EA32A053D878E396ADEEEA7', 'process_name': 'ssm-agent-worker.exe'}, {'pid_hash': '18325228D5FD390F6FEF

The paths come in three forms, corresponding to columns `ptree`, `ptree_list` and `ptree_list_tuples`.

In [8]:
%%sql process_paths <<
select pid_hash, ptree, ptree_list, ptree_list_tuples
from process_path

In [10]:
process_paths

,pid_hash,ptree,ptree_list,ptree_list_tuples
0,00076962A860658B5C74574915AAD94C,=conhost.exe,[00076962A860658B5C74574915AAD94C],"[{'pid_hash': '00076962A860658B5C74574915AAD94C', 'process_name': 'conhost.exe'}]"
1,000C6DD334EAC0C94D4AB0147CB66823,=conhost.exe,[000C6DD334EAC0C94D4AB0147CB66823],"[{'pid_hash': '000C6DD334EAC0C94D4AB0147CB66823', 'process_name': 'conhost.exe'}]"
2,0017CF8579D1B0A203230C4445486C6A,=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe,"[0017CF8579D1B0A203230C4445486C6A, 509D73E59EA32A053D878E396ADEEEA7, 18325228D5FD390F6FEF7E41190D82F3, 200B9DB9449A2F74C1999BD921A616C5, E348B38A38A42696061DD2CE30F52B62, 916C854F45C9A21E9D80F58E975BAB3A]","[{'pid_hash': '0017CF8579D1B0A203230C4445486C6A', 'process_name': 'wmic.exe'}, {'pid_hash': '509D73E59EA32A053D878E396ADEEEA7', 'process_name': 'ssm-agent-worker.exe'}, {'pid_hash': '18325228D5FD390F6FEF7E41190D82F3', 'process_name': 'amazon-ssm-agent.exe'}, {'pid_hash': '200B9DB9449A2F74C1999BD921A616C5', 'process_name': 'services.exe'}, {'pid_hash': 'E348B38A38A42696061DD2CE30F52B62', 'process_name': 'wininit.exe'}, {'pid_hash': '916C854F45C9A21E9D80F58E975BAB3A', 'process_name': 'svchost.exe'}]"
3,001999BCC533CC1B0FE072CA9BB692B9,=conhost.exe,[001999BCC533CC1B0FE072CA9BB692B9],"[{'pid_hash': '001999BCC533CC1B0FE072CA9BB692B9', 'process_name': 'conhost.exe'}]"
4,001FFD625FDE57A058ABD9EA10E208C5,=conhost.exe,[001FFD625FDE57A058ABD9EA10E208C5],"[{'pid_hash': '001FFD625FDE57A058ABD9EA10E208C5', 'process_name': 'conhost.exe'}]"
5,0024EDDD1E7D68B26787944479F7B056,=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe,"[0024EDDD1E7D68B26787944479F7B056, 509D73E59EA32A053D878E396ADEEEA7, 18325228D5FD390F6FEF7E41190D82F3, 200B9DB9449A2F74C1999BD921A616C5, E348B38A38A42696061DD2CE30F52B62, 916C854F45C9A21E9D80F58E975BAB3A]","[{'pid_hash': '0024EDDD1E7D68B26787944479F7B056', 'process_name': 'wmic.exe'}, {'pid_hash': '509D73E59EA32A053D878E396ADEEEA7', 'process_name': 'ssm-agent-worker.exe'}, {'pid_hash': '18325228D5FD390F6FEF7E41190D82F3', 'process_name': 'amazon-ssm-agent.exe'}, {'pid_hash': '200B9DB9449A2F74C1999BD921A616C5', 'process_name': 'services.exe'}, {'pid_hash': 'E348B38A38A42696061DD2CE30F52B62', 'process_name': 'wininit.exe'}, {'pid_hash': '916C854F45C9A21E9D80F58E975BAB3A', 'process_name': 'svchost.exe'}]"
6,00298EB1C6BC14848306E4670ED2B9E4,=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe,"[00298EB1C6BC14848306E4670ED2B9E4, 509D73E59EA32A053D878E396ADEEEA7, 18325228D5FD390F6FEF7E41190D82F3, 200B9DB9449A2F74C1999BD921A616C5, E348B38A38A42696061DD2CE30F52B62, 916C854F45C9A21E9D80F58E975BAB3A]","[{'pid_hash': '00298EB1C6BC14848306E4670ED2B9E4', 'process_name': 'wmic.exe'}, {'pid_hash': '509D73E59EA32A053D878E396ADEEEA7', 'process_name': 'ssm-agent-worker.exe'}, {'pid_hash': '18325228D5FD390F6FEF7E41190D82F3', 'process_name': 'amazon-ssm-agent.exe'}, {'pid_hash': '200B9DB9449A2F74C1999BD921A616C5', 'process_name': 'services.exe'}, {'pid_hash': 'E348B38A38A42696061DD2CE30F52B62', 'process_name': 'wininit.exe'}, {'pid_hash': '916C854F45C9A21E9D80F58E975BAB3A', 'process_name': 'svchost.exe'}]"
7,00398AF501769695839D8C0FDA9FFB36,=conhost.exe,[00398AF501769695839D8C0FDA9FFB36],"[{'pid_hash': '00398AF501769695839D8C0FDA9FFB36', 'process_name': 'conhost.exe'}]"
8,004072560E9BBCB5D8444846D3191CC9,=conhost.exe,[004072560E9BBCB5D8444846D3191CC9],"[{'pid_hash': '004072560E9BBCB5D8444846D3191CC9', 'process_name': 'conhost.exe'}]"
9,0048AAA3DB863E49FDFF040806EFFE4D,=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe,"[0048AAA3DB863E49FDFF040806EFFE4D, 509D73E59EA32A053D878E396ADEEEA7, 18325228D5FD390F6FEF7E41190D82F3, 200B9DB9449A2F74C1999BD921A616C5, E348B38A38A42696061DD2CE30F52B62, 916C854F45C9A21E9D80F58E975BAB3A]","[{'pid_hash': '0048AAA3DB863E49FDFF040806EFFE4D', 'process_name': 'wmic.exe'}, {'pid_hash': '

In [13]:
example = process_paths.loc[process_paths["ptree_list"].map(len) > 4].iloc[0]
example

pid_hash                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   0017CF8579D1B0A203230C4445486C6A
ptree                                                                                                                                                                                                                                                                                                                                                                                                                                         =wmic.exe->ssm-agent-worker.exe->amazon-ssm-ag

The `ptree_list` form shows the path as a sequence of pid-hashes going from the process itself to its known parent closest to the root of the tree.

In [15]:
example["ptree_list"], example["ptree_list"][0] == example["pid_hash"]

(array(['0017CF8579D1B0A203230C4445486C6A',
        '509D73E59EA32A053D878E396ADEEEA7',
        '18325228D5FD390F6FEF7E41190D82F3',
        '200B9DB9449A2F74C1999BD921A616C5',
        'E348B38A38A42696061DD2CE30F52B62',
        '916C854F45C9A21E9D80F58E975BAB3A'], dtype=object),
 True)

The `ptree_list_tuples` recapitulates the same information,
but caches along the `process_name` known for the associated pid-hashes.

In [17]:
df_example = pd.DataFrame([(pair["pid_hash"], pair["process_name"]) for pair in example["ptree_list_tuples"]], columns=["pid_hash", "process_name"])
df_example

,pid_hash,process_name
0,0017CF8579D1B0A203230C4445486C6A,wmic.exe
1,509D73E59EA32A053D878E396ADEEEA7,ssm-agent-worker.exe
2,18325228D5FD390F6FEF7E41190D82F3,amazon-ssm-agent.exe
3,200B9DB9449A2F74C1999BD921A616C5,services.exe
4,E348B38A38A42696061DD2CE30F52B62,wininit.exe
5,916C854F45C9A21E9D80F58E975BAB3A,svchost.exe


In [19]:
%%sql
select EX.pid_hash, EX.process_name as "example:process_name", P.process_name as "process:process_name"
from df_example as EX
inner join process as P using (pid_hash)

,pid_hash,example:process_name,process:process_name
0,0017CF8579D1B0A203230C4445486C6A,wmic.exe,wmic.exe
1,18325228D5FD390F6FEF7E41190D82F3,amazon-ssm-agent.exe,amazon-ssm-agent.exe
2,200B9DB9449A2F74C1999BD921A616C5,services.exe,services.exe
3,916C854F45C9A21E9D80F58E975BAB3A,svchost.exe,svchost.exe
4,E348B38A38A42696061DD2CE30F52B62,wininit.exe,wininit.exe
5,509D73E59EA32A053D878E396ADEEEA7,ssm-agent-worker.exe,ssm-agent-worker.exe


Finally, the `ptree` column projects the sequence of pid-hashes through the `process_name` column as a map.
The resulting strings are concatenated with `->` arrows,
and prepended with `=` sigil.

In [20]:
example["ptree"]

'=wmic.exe->ssm-agent-worker.exe->amazon-ssm-agent.exe->services.exe->wininit.exe->svchost.exe'

Regarding surprises with respect to expectations...
The Windows boot process is much murkier than Linux'.
In principle, process `wininit.exe` is [started by `smss.exe`](https://0xcybery.github.io/blog/Core-Processes-In-Windows-System),
which then terminates.
Thus, the OS gets tasked with reparenting `wininit.exe`...

In [34]:
%%sql
select P.pid_hash, process_path, os_pid, parent_os_pid
from df_example
inner join process as P using (pid_hash)

,pid_hash,process_path,os_pid,parent_os_pid
0,0017CF8579D1B0A203230C4445486C6A,c:\windows\system32\wbem\wmic.exe,7696,4448
1,916C854F45C9A21E9D80F58E975BAB3A,c:\windows\system32\svchost.exe,588,820
2,E348B38A38A42696061DD2CE30F52B62,c:\windows\system32\wininit.exe,684,588
3,509D73E59EA32A053D878E396ADEEEA7,c:\program files\amazon\ssm\ssm-agent-worker.exe,4448,2284
4,200B9DB9449A2F74C1999BD921A616C5,c:\windows\system32\services.exe,820,684
5,18325228D5FD390F6FEF7E41190D82F3,c:\program files\amazon\ssm\amazon-ssm-agent.exe,2284,820


... and makes it a child of a _service process_!?
This makes no sense at all,
and likely points at sensor noise.